# 04. Tournament Simulator

In [ ]:
# %% [markdown]
# # 04. Tournament Simulator
#
# Objetivos:
# - reconstruir estados históricos sin fuga de informacion
# - calibrar un motor Poisson para resultados
# - resolver playoffs UEFA e interconfederacionales
# - completar la plantilla de grupos del Mundial 2026
# - simular fase de grupos
# - simular fase eliminatoria
# - estimar probabilidades por Monte Carlo
#
# Notas:
# - La plantilla de grupos se define localmente, sin depender de groupsPath.
# - Los placeholders de play-offs siguen la estructura oficial de FIFA.
# - El bracket de Round of 32 se implementa como un bracket genericamente ordenado
#   por rendimiento de fase de grupos. Si luego quieres una replica exacta del
#   bracket oficial, se reemplaza solo esa funcion.

# %%
import os
from copy import deepcopy
from pathlib import Path
from collections import defaultdict, deque

import numpy as np
import pandas as pd
from scipy.stats import poisson
from sklearn.linear_model import PoissonRegressor
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, log_loss
from sklearn.preprocessing import StandardScaler

os.environ["LOKY_MAX_CPU_COUNT"] = "1"
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)

cleanPath = Path("../data/clean/clean_matches.csv")
outputDir = Path("../data/output")
#outputDir.mkdir(parents=True, exist_ok=True)

print("Clean path:", cleanPath)
print("Output dir:", outputDir)

Clean path: ..\data\clean\clean_matches.csv
Output dir: ..\data\output


## 1. Plantilla local de grupos

In [3]:
# 
#
# Basada en el sorteo final oficial de FIFA: grupos A-L con seis placeholders pendientes
# (4 UEFA y 2 interconfederacionales).

# %%
groupsTemplate = {
    "A": ["Mexico", "South Korea", "South Africa", "UEFA Playoff D winner"],
    "B": ["Canada", "Switzerland", "Qatar", "UEFA Playoff A winner"],
    "C": ["Brazil", "Morocco", "Scotland", "Haiti"],
    "D": ["USA", "Paraguay", "Australia", "UEFA Playoff C winner"],
    "E": ["Germany", "Ecuador", "Ivory Coast", "Curaçao"],
    "F": ["Netherlands", "Japan", "Tunisia", "UEFA Playoff B winner"],
    "G": ["Belgium", "Iran", "Egypt", "New Zealand"],
    "H": ["Spain", "Uruguay", "Saudi Arabia", "Cape Verde"],
    "I": ["France", "Senegal", "Norway", "Interconfed Playoff 2 winner"],
    "J": ["Argentina", "Austria", "Algeria", "Jordan"],
    "K": ["Portugal", "Colombia", "Uzbekistan", "Interconfed Playoff 1 winner"],
    "L": ["England", "Croatia", "Panama", "Ghana"],
}

print("Grupos plantilla cargados:", {k: len(v) for k, v in groupsTemplate.items()})


Grupos plantilla cargados: {'A': 4, 'B': 4, 'C': 4, 'D': 4, 'E': 4, 'F': 4, 'G': 4, 'H': 4, 'I': 4, 'J': 4, 'K': 4, 'L': 4}


## 2. Play-offs oficiales

In [4]:
# ## 2. Play-offs oficiales
#
# UEFA:
# - Path A: Italy / Northern Ireland, Wales / Bosnia and Herzegovina
# - Path B: Ukraine / Sweden, Poland / Albania
# - Path C: Turkey / Romania, Slovakia / Kosovo
# - Path D: Denmark / North Macedonia, Czechia / Republic of Ireland
#
# Interconfederacional:
# - Pathway 1: New Caledonia / Jamaica, final vs DR Congo
# - Pathway 2: Bolivia / Suriname, final vs Iraq

# %%
uefaPlayoffPaths = {
    "A": {
        "semi1": ("Italy", "Northern Ireland"),
        "semi2": ("Wales", "Bosnia and Herzegovina"),
    },
    "B": {
        "semi1": ("Ukraine", "Sweden"),
        "semi2": ("Poland", "Albania"),
    },
    "C": {
        "semi1": ("Turkey", "Romania"),
        "semi2": ("Slovakia", "Kosovo"),
    },
    "D": {
        "semi1": ("Denmark", "North Macedonia"),
        "semi2": ("Czechia", "Republic of Ireland"),
    },
}

interconfedPlayoffPaths = {
    "1": {
        "semi": ("New Caledonia", "Jamaica"),
        "finalOpponent": "DR Congo",
    },
    "2": {
        "semi": ("Bolivia", "Suriname"),
        "finalOpponent": "Iraq",
    },
}

## 3. Funciones auxiliares

In [5]:
# ## 3. Funciones auxiliares

# %%
def safeMean(values):
    if len(values) == 0:
        return np.nan
    return float(np.mean(values))

def safeRate(numerator, denominator):
    if denominator == 0:
        return np.nan
    return float(numerator / denominator)

def getResultPoints(homeScore, awayScore):
    if homeScore > awayScore:
        return 3, 0
    if homeScore < awayScore:
        return 0, 3
    return 1, 1

def expectedScore(ratingA, ratingB):
    return 1 / (1 + 10 ** ((ratingB - ratingA) / 400))

def updateElo(homeElo, awayElo, homeScore, awayScore, neutral=False, baseK=20, homeAdvantage=65):
    adjustedHomeElo = homeElo if neutral else homeElo + homeAdvantage
    expectedHome = expectedScore(adjustedHomeElo, awayElo)

    if homeScore > awayScore:
        actualHome = 1.0
    elif homeScore < awayScore:
        actualHome = 0.0
    else:
        actualHome = 0.5

    newHomeElo = homeElo + baseK * (actualHome - expectedHome)
    newAwayElo = awayElo + baseK * ((1 - actualHome) - (1 - expectedHome))
    return newHomeElo, newAwayElo

def createTeamState():
    return {
        "elo": 1500.0,
        "matchesPlayed": 0,
        "wins": 0,
        "draws": 0,
        "losses": 0,
        "goalsFor": 0,
        "goalsAgainst": 0,
        "lastMatchDate": None,
        "last5Points": deque(maxlen=5),
        "last5GoalsFor": deque(maxlen=5),
        "last5GoalsAgainst": deque(maxlen=5),
        "last10Points": deque(maxlen=10),
        "last10GoalsFor": deque(maxlen=10),
        "last10GoalsAgainst": deque(maxlen=10),
    }

def getPairKey(teamA, teamB):
    return tuple(sorted([teamA, teamB]))

def buildHistoricalFeatureDataset(historyDf):
    """
    Reconstruye un dataset pre-partido con estados historicos.
    Retorna:
    - featureDf: dataset de entrenamiento
    - teamStates: snapshot final de estados historicos
    """
    teamState = defaultdict(createTeamState)
    featureRows = []

    for row in historyDf.itertuples(index=False):
        matchDate = row.date
        homeTeam = row.homeTeam
        awayTeam = row.awayTeam
        homeScore = int(row.homeScore)
        awayScore = int(row.awayScore)
        neutralFlag = int(bool(row.neutral))

        homeState = teamState[homeTeam]
        awayState = teamState[awayTeam]

        homeElo = homeState["elo"]
        awayElo = awayState["elo"]
        eloDiff = homeElo - awayElo
        absEloDiff = abs(eloDiff)

        homeMatchesPlayed = homeState["matchesPlayed"]
        awayMatchesPlayed = awayState["matchesPlayed"]

        homeGoalsForAvg = safeRate(homeState["goalsFor"], homeMatchesPlayed)
        awayGoalsForAvg = safeRate(awayState["goalsFor"], awayMatchesPlayed)
        homeGoalsAgainstAvg = safeRate(homeState["goalsAgainst"], homeMatchesPlayed)
        awayGoalsAgainstAvg = safeRate(awayState["goalsAgainst"], awayMatchesPlayed)

        homeGoalDiffAvg = safeRate(homeState["goalsFor"] - homeState["goalsAgainst"], homeMatchesPlayed)
        awayGoalDiffAvg = safeRate(awayState["goalsFor"] - awayState["goalsAgainst"], awayMatchesPlayed)

        homeLast5PointsAvg = safeMean(homeState["last5Points"])
        awayLast5PointsAvg = safeMean(awayState["last5Points"])
        homeLast10PointsAvg = safeMean(homeState["last10Points"])
        awayLast10PointsAvg = safeMean(awayState["last10Points"])

        homeDaysSinceLastMatch = (
            (matchDate - homeState["lastMatchDate"]).days if homeState["lastMatchDate"] is not None else np.nan
        )
        awayDaysSinceLastMatch = (
            (matchDate - awayState["lastMatchDate"]).days if awayState["lastMatchDate"] is not None else np.nan
        )

        totalGoalsAvg = (
            homeGoalsForAvg + awayGoalsForAvg
            if pd.notna(homeGoalsForAvg) and pd.notna(awayGoalsForAvg)
            else np.nan
        )
        lowScoringSignal = 1 if (pd.notna(totalGoalsAvg) and totalGoalsAvg < 2.5) else 0
        drawTendency = absEloDiff * totalGoalsAvg if pd.notna(totalGoalsAvg) else np.nan

        featureRows.append(
            {
                "date": matchDate,
                "homeTeam": homeTeam,
                "awayTeam": awayTeam,
                "neutral": neutralFlag,
                "homeScore": homeScore,
                "awayScore": awayScore,
                "homeElo": homeElo,
                "awayElo": awayElo,
                "eloDiff": eloDiff,
                "absEloDiff": absEloDiff,
                "homeGoalsForAvg": homeGoalsForAvg,
                "awayGoalsForAvg": awayGoalsForAvg,
                "homeGoalsAgainstAvg": homeGoalsAgainstAvg,
                "awayGoalsAgainstAvg": awayGoalsAgainstAvg,
                "homeGoalDiffAvg": homeGoalDiffAvg,
                "awayGoalDiffAvg": awayGoalDiffAvg,
                "homeLast5PointsAvg": homeLast5PointsAvg,
                "awayLast5PointsAvg": awayLast5PointsAvg,
                "homeLast10PointsAvg": homeLast10PointsAvg,
                "awayLast10PointsAvg": awayLast10PointsAvg,
                "homeDaysSinceLastMatch": homeDaysSinceLastMatch,
                "awayDaysSinceLastMatch": awayDaysSinceLastMatch,
                "daysSinceLastMatchDiff": (
                    homeDaysSinceLastMatch - awayDaysSinceLastMatch
                    if pd.notna(homeDaysSinceLastMatch) and pd.notna(awayDaysSinceLastMatch)
                    else np.nan
                ),
                "totalGoalsAvg": totalGoalsAvg,
                "lowScoringSignal": lowScoringSignal,
                "drawTendency": drawTendency,
                "target": "win" if homeScore > awayScore else "draw" if homeScore == awayScore else "loss",
            }
        )

        homePoints, awayPoints = getResultPoints(homeScore, awayScore)

        homeState["matchesPlayed"] += 1
        awayState["matchesPlayed"] += 1
        homeState["wins"] += int(homeScore > awayScore)
        homeState["draws"] += int(homeScore == awayScore)
        homeState["losses"] += int(homeScore < awayScore)
        awayState["wins"] += int(awayScore > homeScore)
        awayState["draws"] += int(homeScore == awayScore)
        awayState["losses"] += int(awayScore < homeScore)

        homeState["goalsFor"] += homeScore
        homeState["goalsAgainst"] += awayScore
        awayState["goalsFor"] += awayScore
        awayState["goalsAgainst"] += homeScore

        homeState["last5Points"].append(homePoints)
        awayState["last5Points"].append(awayPoints)
        homeState["last5GoalsFor"].append(homeScore)
        homeState["last5GoalsAgainst"].append(awayScore)
        awayState["last5GoalsFor"].append(awayScore)
        awayState["last5GoalsAgainst"].append(homeScore)

        homeState["last10Points"].append(homePoints)
        awayState["last10Points"].append(awayPoints)
        homeState["last10GoalsFor"].append(homeScore)
        homeState["last10GoalsAgainst"].append(awayScore)
        awayState["last10GoalsFor"].append(awayScore)
        awayState["last10GoalsAgainst"].append(homeScore)

        homeState["lastMatchDate"] = matchDate
        awayState["lastMatchDate"] = matchDate

        newHomeElo, newAwayElo = updateElo(homeElo, awayElo, homeScore, awayScore, neutral=bool(neutralFlag))
        homeState["elo"] = newHomeElo
        awayState["elo"] = newAwayElo

    featureDf = pd.DataFrame(featureRows).sort_values("date", kind="mergesort").reset_index(drop=True)
    return featureDf, teamState

def buildFutureFeatureRow(homeTeam, awayTeam, teamStates, neutral=True, matchDate=None):
    if matchDate is None:
        matchDate = pd.Timestamp.today()

    if homeTeam not in teamStates:
        teamStates[homeTeam] = createTeamState()
    if awayTeam not in teamStates:
        teamStates[awayTeam] = createTeamState()

    homeState = teamStates[homeTeam]
    awayState = teamStates[awayTeam]

    homeElo = homeState["elo"]
    awayElo = awayState["elo"]
    eloDiff = homeElo - awayElo
    absEloDiff = abs(eloDiff)

    homeMatchesPlayed = homeState["matchesPlayed"]
    awayMatchesPlayed = awayState["matchesPlayed"]

    homeGoalsForAvg = safeRate(homeState["goalsFor"], homeMatchesPlayed)
    awayGoalsForAvg = safeRate(awayState["goalsFor"], awayMatchesPlayed)
    homeGoalsAgainstAvg = safeRate(homeState["goalsAgainst"], homeMatchesPlayed)
    awayGoalsAgainstAvg = safeRate(awayState["goalsAgainst"], awayMatchesPlayed)

    homeGoalDiffAvg = safeRate(homeState["goalsFor"] - homeState["goalsAgainst"], homeMatchesPlayed)
    awayGoalDiffAvg = safeRate(awayState["goalsFor"] - awayState["goalsAgainst"], awayMatchesPlayed)

    homeLast5PointsAvg = safeMean(homeState["last5Points"])
    awayLast5PointsAvg = safeMean(awayState["last5Points"])
    homeLast10PointsAvg = safeMean(homeState["last10Points"])
    awayLast10PointsAvg = safeMean(awayState["last10Points"])

    homeDaysSinceLastMatch = (
        (matchDate - homeState["lastMatchDate"]).days if homeState["lastMatchDate"] is not None else np.nan
    )
    awayDaysSinceLastMatch = (
        (matchDate - awayState["lastMatchDate"]).days if awayState["lastMatchDate"] is not None else np.nan
    )

    totalGoalsAvg = (
        homeGoalsForAvg + awayGoalsForAvg
        if pd.notna(homeGoalsForAvg) and pd.notna(awayGoalsForAvg)
        else np.nan
    )
    lowScoringSignal = 1 if (pd.notna(totalGoalsAvg) and totalGoalsAvg < 2.5) else 0
    drawTendency = absEloDiff * totalGoalsAvg if pd.notna(totalGoalsAvg) else np.nan

    row = {
        "neutral": int(bool(neutral)),
        "homeElo": homeElo,
        "awayElo": awayElo,
        "eloDiff": eloDiff,
        "absEloDiff": absEloDiff,
        "homeGoalsForAvg": homeGoalsForAvg,
        "awayGoalsForAvg": awayGoalsForAvg,
        "homeGoalsAgainstAvg": homeGoalsAgainstAvg,
        "awayGoalsAgainstAvg": awayGoalsAgainstAvg,
        "homeGoalDiffAvg": homeGoalDiffAvg,
        "awayGoalDiffAvg": awayGoalDiffAvg,
        "homeLast5PointsAvg": homeLast5PointsAvg,
        "awayLast5PointsAvg": awayLast5PointsAvg,
        "homeLast10PointsAvg": homeLast10PointsAvg,
        "awayLast10PointsAvg": awayLast10PointsAvg,
        "homeDaysSinceLastMatch": homeDaysSinceLastMatch,
        "awayDaysSinceLastMatch": awayDaysSinceLastMatch,
        "daysSinceLastMatchDiff": (
            homeDaysSinceLastMatch - awayDaysSinceLastMatch
            if pd.notna(homeDaysSinceLastMatch) and pd.notna(awayDaysSinceLastMatch)
            else np.nan
        ),
        "totalGoalsAvg": totalGoalsAvg,
        "lowScoringSignal": lowScoringSignal,
        "drawTendency": drawTendency,
    }

    return pd.DataFrame([row])

def applyMatchToStates(teamStates, homeTeam, awayTeam, homeGoals, awayGoals, matchDate, neutral=True):
    if homeTeam not in teamStates:
        teamStates[homeTeam] = createTeamState()
    if awayTeam not in teamStates:
        teamStates[awayTeam] = createTeamState()

    homeState = teamStates[homeTeam]
    awayState = teamStates[awayTeam]

    homePoints, awayPoints = getResultPoints(homeGoals, awayGoals)

    homeState["matchesPlayed"] += 1
    awayState["matchesPlayed"] += 1
    homeState["wins"] += int(homeGoals > awayGoals)
    homeState["draws"] += int(homeGoals == awayGoals)
    homeState["losses"] += int(homeGoals < awayGoals)
    awayState["wins"] += int(awayGoals > homeGoals)
    awayState["draws"] += int(homeGoals == awayGoals)
    awayState["losses"] += int(awayGoals < homeGoals)

    homeState["goalsFor"] += homeGoals
    homeState["goalsAgainst"] += awayGoals
    awayState["goalsFor"] += awayGoals
    awayState["goalsAgainst"] += homeGoals

    homeState["last5Points"].append(homePoints)
    awayState["last5Points"].append(awayPoints)
    homeState["last5GoalsFor"].append(homeGoals)
    homeState["last5GoalsAgainst"].append(awayGoals)
    awayState["last5GoalsFor"].append(awayGoals)
    awayState["last5GoalsAgainst"].append(homeGoals)

    homeState["last10Points"].append(homePoints)
    awayState["last10Points"].append(awayPoints)
    homeState["last10GoalsFor"].append(homeGoals)
    homeState["last10GoalsAgainst"].append(awayGoals)
    awayState["last10GoalsFor"].append(awayGoals)
    awayState["last10GoalsAgainst"].append(homeGoals)

    homeState["lastMatchDate"] = matchDate
    awayState["lastMatchDate"] = matchDate

    newHomeElo, newAwayElo = updateElo(
        homeState["elo"],
        awayState["elo"],
        homeGoals,
        awayGoals,
        neutral=neutral,
    )
    homeState["elo"] = newHomeElo
    awayState["elo"] = newAwayElo

## 4. Carga historica y construccion de estados

In [6]:
# ## 4. Carga historica y construccion de estados

# %%
historyDf = pd.read_csv(cleanPath)
historyDf["date"] = pd.to_datetime(historyDf["date"])
historyDf = historyDf.sort_values("date", kind="mergesort").reset_index(drop=True)

historicalDf, historicalStates = buildHistoricalFeatureDataset(historyDf)

print("Shape historicalDf:", historicalDf.shape)
historicalDf.head()

# %%
nullSummary = historicalDf.isnull().mean().sort_values(ascending=False)
print(nullSummary[nullSummary > 0].head(20))

# %%
numericColumns = historicalDf.select_dtypes(include=[np.number]).columns.tolist()
for col in numericColumns:
    if historicalDf[col].isnull().any():
        historicalDf[col] = historicalDf[col].fillna(historicalDf[col].median())

print("Nulos restantes:", historicalDf.isnull().sum().sum())

# %%
targetMapping = {"loss": 0, "draw": 1, "win": 2}
historicalDf["targetEncoded"] = historicalDf["target"].map(targetMapping)

assert historicalDf["targetEncoded"].notnull().all(), "Hay targetEncoded nulos"

# %%
featureCols = [
    "neutral",
    "homeElo",
    "awayElo",
    "eloDiff",
    "absEloDiff",
    "homeGoalsForAvg",
    "awayGoalsForAvg",
    "homeGoalsAgainstAvg",
    "awayGoalsAgainstAvg",
    "homeGoalDiffAvg",
    "awayGoalDiffAvg",
    "homeLast5PointsAvg",
    "awayLast5PointsAvg",
    "homeLast10PointsAvg",
    "awayLast10PointsAvg",
    "homeDaysSinceLastMatch",
    "awayDaysSinceLastMatch",
    "daysSinceLastMatchDiff",
    "totalGoalsAvg",
    "lowScoringSignal",
    "drawTendency",
]

assert not [c for c in featureCols if c not in historicalDf.columns], "Faltan features en historicalDf"


Shape historicalDf: (25013, 27)
daysSinceLastMatchDiff    0.010515
drawTendency              0.010515
totalGoalsAvg             0.010515
homeLast5PointsAvg        0.006397
homeLast10PointsAvg       0.006397
homeGoalDiffAvg           0.006397
homeDaysSinceLastMatch    0.006397
homeGoalsAgainstAvg       0.006397
homeGoalsForAvg           0.006397
awayGoalDiffAvg           0.006317
awayLast5PointsAvg        0.006317
awayDaysSinceLastMatch    0.006317
awayLast10PointsAvg       0.006317
awayGoalsForAvg           0.006317
awayGoalsAgainstAvg       0.006317
dtype: float64
Nulos restantes: 0


## 5. Split temporal para calibracion del motor Poisson

In [7]:
# ## 5. Split temporal para calibracion del motor Poisson

# %%
trainEnd = int(len(historicalDf) * 0.70)
valEnd = trainEnd + int(len(historicalDf) * 0.15)

trainDf = historicalDf.iloc[:trainEnd].copy()
valDf = historicalDf.iloc[trainEnd:valEnd].copy()
testDf = historicalDf.iloc[valEnd:].copy()

print("Train:", trainDf.shape)
print("Val:", valDf.shape)
print("Test:", testDf.shape)

X_train = trainDf[featureCols].copy()
X_val = valDf[featureCols].copy()
X_test = testDf[featureCols].copy()

y_home_train = trainDf["homeScore"].to_numpy()
y_away_train = trainDf["awayScore"].to_numpy()

y_home_val = valDf["homeScore"].to_numpy()
y_away_val = valDf["awayScore"].to_numpy()

y_home_test = testDf["homeScore"].to_numpy()
y_away_test = testDf["awayScore"].to_numpy()

y_match_val = np.where(valDf["homeScore"] > valDf["awayScore"], 2, np.where(valDf["homeScore"] < valDf["awayScore"], 0, 1))
y_match_test = np.where(testDf["homeScore"] > testDf["awayScore"], 2, np.where(testDf["homeScore"] < testDf["awayScore"], 0, 1))

print("Numero de features para simulacion:", len(featureCols))
print(pd.Series(y_match_val).value_counts(normalize=True).sort_index())

Train: (17509, 28)
Val: (3751, 28)
Test: (3753, 28)
Numero de features para simulacion: 21
0    0.291389
1    0.227939
2    0.480672
Name: proportion, dtype: float64


# 6. TUNING DEL POISSON

In [8]:
# =========================================================
# 6. TUNING DEL POISSON
# =========================================================
poissonScaler = StandardScaler()

X_train_scaled = poissonScaler.fit_transform(X_train)
X_val_scaled = poissonScaler.transform(X_val)
X_test_scaled = poissonScaler.transform(X_test)

def outcomeProbabilitiesFromLambdas(lh, la, drawBoost=1.0, maxGoals=10):
    goals = np.arange(maxGoals + 1)
    pHome = poisson.pmf(goals, lh)
    pAway = poisson.pmf(goals, la)
    grid = np.outer(pHome, pAway)

    diag_idx = np.arange(grid.shape[0])
    grid[diag_idx, diag_idx] *= drawBoost

    total = grid.sum()
    if total > 0:
        grid = grid / total

    return np.array([
        np.triu(grid, 1).sum(),  # loss
        np.trace(grid),          # draw
        np.tril(grid, -1).sum(), # win
    ])

def predictOutcomeProba(lambdaHomeArr, lambdaAwayArr, drawBoost=1.0, maxGoals=10):
    return np.array([
        outcomeProbabilitiesFromLambdas(lh, la, drawBoost=drawBoost, maxGoals=maxGoals)
        for lh, la in zip(lambdaHomeArr, lambdaAwayArr)
    ])

alphaGrid = [0.01, 0.05, 0.1, 0.3, 0.5, 1.0]
drawBoostGrid = [1.0, 1.1, 1.2, 1.3, 1.4, 1.5]

bestConfig = {
    "alpha": 0.1,
    "drawBoost": 1.0,
    "logLoss": np.inf,
    "balancedAccuracy": -np.inf,
    "f1": -np.inf,
}

for alpha in alphaGrid:
    candidateHome = PoissonRegressor(alpha=alpha, max_iter=1000)
    candidateAway = PoissonRegressor(alpha=alpha, max_iter=1000)

    candidateHome.fit(X_train_scaled, y_home_train)
    candidateAway.fit(X_train_scaled, y_away_train)

    lambdaHome_val = np.clip(candidateHome.predict(X_val_scaled), 0.01, None)
    lambdaAway_val = np.clip(candidateAway.predict(X_val_scaled), 0.01, None)

    for drawBoost in drawBoostGrid:
        valProba = predictOutcomeProba(lambdaHome_val, lambdaAway_val, drawBoost=drawBoost, maxGoals=10)
        valPred = np.argmax(valProba, axis=1)

        currentLogLoss = log_loss(y_match_val, valProba, labels=[0, 1, 2])
        currentBalAcc = balanced_accuracy_score(y_match_val, valPred)
        currentF1 = f1_score(y_match_val, valPred, average="macro")

        if currentLogLoss < bestConfig["logLoss"]:
            bestConfig = {
                "alpha": alpha,
                "drawBoost": drawBoost,
                "logLoss": currentLogLoss,
                "balancedAccuracy": currentBalAcc,
                "f1": currentF1,
            }

print("Best Poisson config:", bestConfig)

Best Poisson config: {'alpha': 0.01, 'drawBoost': 1.1, 'logLoss': 0.8594522631769036, 'balancedAccuracy': np.float64(0.5018515614837998), 'f1': 0.441755655511443}


# 7. FIT FINAL DEL MOTOR POISSON

In [9]:
# =========================================================
# 7. FIT FINAL DEL MOTOR POISSON
# =========================================================
X_trainVal = pd.concat([X_train, X_val], axis=0)
y_home_trainVal = pd.concat([trainDf["homeScore"], valDf["homeScore"]], axis=0).to_numpy()
y_away_trainVal = pd.concat([trainDf["awayScore"], valDf["awayScore"]], axis=0).to_numpy()

scalerFinal = StandardScaler()
X_trainVal_scaled = scalerFinal.fit_transform(X_trainVal)
X_test_scaled_final = scalerFinal.transform(X_test)

finalPoissonHome = PoissonRegressor(alpha=bestConfig["alpha"], max_iter=1000)
finalPoissonAway = PoissonRegressor(alpha=bestConfig["alpha"], max_iter=1000)

finalPoissonHome.fit(X_trainVal_scaled, y_home_trainVal)
finalPoissonAway.fit(X_trainVal_scaled, y_away_trainVal)

lambdaHome_test = np.clip(finalPoissonHome.predict(X_test_scaled_final), 0.01, None)
lambdaAway_test = np.clip(finalPoissonAway.predict(X_test_scaled_final), 0.01, None)

testProba = predictOutcomeProba(
    lambdaHome_test,
    lambdaAway_test,
    drawBoost=bestConfig["drawBoost"],
    maxGoals=10,
)
testPred = np.argmax(testProba, axis=1)

print("\nPOISSON SIMULATION MODEL (TEST)")
print("Accuracy:", accuracy_score(y_match_test, testPred))
print("Balanced Accuracy:", balanced_accuracy_score(y_match_test, testPred))
print("F1:", f1_score(y_match_test, testPred, average="macro"))
print("LogLoss:", log_loss(y_match_test, testProba, labels=[0, 1, 2]))
print()
print(pd.Series(y_match_test).value_counts(normalize=True).sort_index())



POISSON SIMULATION MODEL (TEST)
Accuracy: 0.6016520117239542
Balanced Accuracy: 0.5018096773651769
F1: 0.4406869911846019
LogLoss: 0.8775548645691255

0    0.297096
1    0.228617
2    0.474287
Name: proportion, dtype: float64


# 8. CURRENT STATES SNAPSHOT FOR FUTURE SIMULATIONS

In [10]:
# =========================================================
# 8. CURRENT STATES SNAPSHOT FOR FUTURE SIMULATIONS
# =========================================================
# Usamos el estado historico final ya reconstruido.
currentStates = deepcopy(historicalStates)

# %%
def buildFutureFeatureRow(homeTeam, awayTeam, teamStates, neutral=True, matchDate=None):
    if matchDate is None:
        matchDate = historyDf["date"].max() + pd.Timedelta(days=1)

    if homeTeam not in teamStates:
        teamStates[homeTeam] = createTeamState()
    if awayTeam not in teamStates:
        teamStates[awayTeam] = createTeamState()

    homeState = teamStates[homeTeam]
    awayState = teamStates[awayTeam]

    homeElo = homeState["elo"]
    awayElo = awayState["elo"]
    eloDiff = homeElo - awayElo
    absEloDiff = abs(eloDiff)

    homeMatchesPlayed = homeState["matchesPlayed"]
    awayMatchesPlayed = awayState["matchesPlayed"]

    homeGoalsForAvg = safeRate(homeState["goalsFor"], homeMatchesPlayed)
    awayGoalsForAvg = safeRate(awayState["goalsFor"], awayMatchesPlayed)
    homeGoalsAgainstAvg = safeRate(homeState["goalsAgainst"], homeMatchesPlayed)
    awayGoalsAgainstAvg = safeRate(awayState["goalsAgainst"], awayMatchesPlayed)

    homeGoalDiffAvg = safeRate(homeState["goalsFor"] - homeState["goalsAgainst"], homeMatchesPlayed)
    awayGoalDiffAvg = safeRate(awayState["goalsFor"] - awayState["goalsAgainst"], awayMatchesPlayed)

    homeLast5PointsAvg = safeMean(homeState["last5Points"])
    awayLast5PointsAvg = safeMean(awayState["last5Points"])
    homeLast10PointsAvg = safeMean(homeState["last10Points"])
    awayLast10PointsAvg = safeMean(awayState["last10Points"])

    homeDaysSinceLastMatch = (
        (matchDate - homeState["lastMatchDate"]).days if homeState["lastMatchDate"] is not None else np.nan
    )
    awayDaysSinceLastMatch = (
        (matchDate - awayState["lastMatchDate"]).days if awayState["lastMatchDate"] is not None else np.nan
    )

    totalGoalsAvg = (
        homeGoalsForAvg + awayGoalsForAvg
        if pd.notna(homeGoalsForAvg) and pd.notna(awayGoalsForAvg)
        else np.nan
    )

    lowScoringSignal = 1 if (pd.notna(totalGoalsAvg) and totalGoalsAvg < 2.5) else 0
    drawTendency = absEloDiff * totalGoalsAvg if pd.notna(totalGoalsAvg) else np.nan

    row = {
        "neutral": int(bool(neutral)),
        "homeElo": homeElo,
        "awayElo": awayElo,
        "eloDiff": eloDiff,
        "absEloDiff": absEloDiff,
        "homeGoalsForAvg": homeGoalsForAvg,
        "awayGoalsForAvg": awayGoalsForAvg,
        "homeGoalsAgainstAvg": homeGoalsAgainstAvg,
        "awayGoalsAgainstAvg": awayGoalsAgainstAvg,
        "homeGoalDiffAvg": homeGoalDiffAvg,
        "awayGoalDiffAvg": awayGoalDiffAvg,
        "homeLast5PointsAvg": homeLast5PointsAvg,
        "awayLast5PointsAvg": awayLast5PointsAvg,
        "homeLast10PointsAvg": homeLast10PointsAvg,
        "awayLast10PointsAvg": awayLast10PointsAvg,
        "homeDaysSinceLastMatch": homeDaysSinceLastMatch,
        "awayDaysSinceLastMatch": awayDaysSinceLastMatch,
        "daysSinceLastMatchDiff": (
            homeDaysSinceLastMatch - awayDaysSinceLastMatch
            if pd.notna(homeDaysSinceLastMatch) and pd.notna(awayDaysSinceLastMatch)
            else np.nan
        ),
        "totalGoalsAvg": totalGoalsAvg,
        "lowScoringSignal": lowScoringSignal,
        "drawTendency": drawTendency,
    }

    rowDf = pd.DataFrame([row])
    rowDf = rowDf.reindex(columns=featureCols, fill_value=np.nan)

    for col in featureCols:
        if col not in rowDf.columns:
            rowDf[col] = np.nan

    return rowDf[featureCols]

def simulateMatch(homeTeam, awayTeam, teamStates, scaler, poissonHomeModel, poissonAwayModel, drawBoost=1.0, neutral=True, matchDate=None, rng=None):
    if rng is None:
        rng = np.random.default_rng()

    featureRow = buildFutureFeatureRow(homeTeam, awayTeam, teamStates, neutral=neutral, matchDate=matchDate)
    featureRow = featureRow.fillna(pd.Series(X_trainVal.median(), index=featureCols))
    featureScaled = scaler.transform(featureRow)

    lambdaHome = float(np.clip(poissonHomeModel.predict(featureScaled)[0], 0.01, None))
    lambdaAway = float(np.clip(poissonAwayModel.predict(featureScaled)[0], 0.01, None))

    goals = np.arange(11)
    pHome = poisson.pmf(goals, lambdaHome)
    pAway = poisson.pmf(goals, lambdaAway)
    grid = np.outer(pHome, pAway)

    diag_idx = np.arange(grid.shape[0])
    grid[diag_idx, diag_idx] *= drawBoost
    grid = grid / grid.sum()

    flat = grid.ravel()
    idx = rng.choice(len(flat), p=flat)
    homeGoals, awayGoals = np.unravel_index(idx, grid.shape)

    if homeGoals > awayGoals:
        outcome = "homeWin"
    elif homeGoals < awayGoals:
        outcome = "awayWin"
    else:
        outcome = "draw"

    outcomeProba = np.array([
        np.triu(grid, 1).sum(),
        np.trace(grid),
        np.tril(grid, -1).sum(),
    ])

    applyMatchToStates(
        teamStates=teamStates,
        homeTeam=homeTeam,
        awayTeam=awayTeam,
        homeGoals=int(homeGoals),
        awayGoals=int(awayGoals),
        matchDate=matchDate if matchDate is not None else pd.Timestamp.today(),
        neutral=neutral,
    )

    return {
        "homeTeam": homeTeam,
        "awayTeam": awayTeam,
        "homeGoals": int(homeGoals),
        "awayGoals": int(awayGoals),
        "outcome": outcome,
        "probaLoss": float(outcomeProba[0]),
        "probaDraw": float(outcomeProba[1]),
        "probaWin": float(outcomeProba[2]),
        "lambdaHome": lambdaHome,
        "lambdaAway": lambdaAway,
    }

# 9. RESOLUCION DE PLAY-OFFS

In [11]:
# =========================================================
# 9. RESOLUCION DE PLAY-OFFS
# =========================================================
def simulateUEFAPlayoffs(teamStates, scaler, poissonHomeModel, poissonAwayModel, drawBoost, rng=None, matchDate=None):
    if rng is None:
        rng = np.random.default_rng()

    winners = {}

    for pathName, path in uefaPlayoffPaths.items():
        semi1Home, semi1Away = path["semi1"]
        semi2Home, semi2Away = path["semi2"]

        semi1 = simulateMatch(
            semi1Home, semi1Away, teamStates, scaler, poissonHomeModel, poissonAwayModel,
            drawBoost=drawBoost, neutral=False, matchDate=matchDate, rng=rng
        )
        semi2 = simulateMatch(
            semi2Home, semi2Away, teamStates, scaler, poissonHomeModel, poissonAwayModel,
            drawBoost=drawBoost, neutral=False, matchDate=matchDate, rng=rng
        )

        finalMatch = simulateMatch(
            semi1["homeTeam"] if semi1["outcome"] == "homeWin" else semi1["awayTeam"],
            semi2["homeTeam"] if semi2["outcome"] == "homeWin" else semi2["awayTeam"],
            teamStates, scaler, poissonHomeModel, poissonAwayModel,
            drawBoost=drawBoost, neutral=False, matchDate=matchDate, rng=rng
        )

        winners[pathName] = finalMatch["homeTeam"] if finalMatch["outcome"] == "homeWin" else finalMatch["awayTeam"]

    return winners

def simulateInterconfedPlayoffs(teamStates, scaler, poissonHomeModel, poissonAwayModel, drawBoost, rng=None, matchDate=None):
    if rng is None:
        rng = np.random.default_rng()

    winners = {}

    for pathName, path in interconfedPlayoffPaths.items():
        semiHome, semiAway = path["semi"]
        finalOpponent = path["finalOpponent"]

        semi = simulateMatch(
            semiHome, semiAway, teamStates, scaler, poissonHomeModel, poissonAwayModel,
            drawBoost=drawBoost, neutral=True, matchDate=matchDate, rng=rng
        )
        semiWinner = semi["homeTeam"] if semi["outcome"] == "homeWin" else semi["awayTeam"]

        finalMatch = simulateMatch(
            semiWinner, finalOpponent, teamStates, scaler, poissonHomeModel, poissonAwayModel,
            drawBoost=drawBoost, neutral=True, matchDate=matchDate, rng=rng
        )

        winners[pathName] = finalMatch["homeTeam"] if finalMatch["outcome"] == "homeWin" else finalMatch["awayTeam"]

    return winners

def resolveGroupsTemplate(groupsTemplate, uefaWinners, interconfedWinners):
    resolved = deepcopy(groupsTemplate)

    placeholderMap = {
        "UEFA Playoff A winner": uefaWinners["A"],
        "UEFA Playoff B winner": uefaWinners["B"],
        "UEFA Playoff C winner": uefaWinners["C"],
        "UEFA Playoff D winner": uefaWinners["D"],
        "Interconfed Playoff 1 winner": interconfedWinners["1"],
        "Interconfed Playoff 2 winner": interconfedWinners["2"],
    }

    for groupName, teams in resolved.items():
        resolved[groupName] = [placeholderMap.get(team, team) for team in teams]

    return resolved

# 10. SIMULACION DE FASE DE GRUPOS

In [12]:
# =========================================================
# 10. SIMULACION DE FASE DE GRUPOS
# =========================================================
def rankStandings(standingsDict):
    ranked = sorted(
        standingsDict.values(),
        key=lambda x: (x["points"], x["gd"], x["gf"], x["wins"]),
        reverse=True,
    )
    return ranked

def simulateGroupStage(groups, teamStates, scaler, poissonHomeModel, poissonAwayModel, drawBoost, rng=None, matchDate=None):
    if rng is None:
        rng = np.random.default_rng()

    groupResults = {}

    for groupName, teams in groups.items():
        standings = {
            team: {
                "team": team,
                "group": groupName,
                "points": 0,
                "played": 0,
                "wins": 0,
                "draws": 0,
                "losses": 0,
                "gf": 0,
                "ga": 0,
                "gd": 0,
            }
            for team in teams
        }

        for i in range(len(teams)):
            for j in range(i + 1, len(teams)):
                homeTeam = teams[i]
                awayTeam = teams[j]
                match = simulateMatch(
                    homeTeam, awayTeam, teamStates, scaler, poissonHomeModel, poissonAwayModel,
                    drawBoost=drawBoost, neutral=True, matchDate=matchDate, rng=rng
                )

                hg = match["homeGoals"]
                ag = match["awayGoals"]

                standings[homeTeam]["played"] += 1
                standings[awayTeam]["played"] += 1
                standings[homeTeam]["gf"] += hg
                standings[homeTeam]["ga"] += ag
                standings[awayTeam]["gf"] += ag
                standings[awayTeam]["ga"] += hg

                if hg > ag:
                    standings[homeTeam]["points"] += 3
                    standings[homeTeam]["wins"] += 1
                    standings[awayTeam]["losses"] += 1
                elif hg < ag:
                    standings[awayTeam]["points"] += 3
                    standings[awayTeam]["wins"] += 1
                    standings[homeTeam]["losses"] += 1
                else:
                    standings[homeTeam]["points"] += 1
                    standings[awayTeam]["points"] += 1
                    standings[homeTeam]["draws"] += 1
                    standings[awayTeam]["draws"] += 1

        for team in teams:
            standings[team]["gd"] = standings[team]["gf"] - standings[team]["ga"]

        ranked = rankStandings(standings)
        groupResults[groupName] = ranked

    return groupResults

# 11. SELECCION DE CLASIFICADOS

In [13]:
# =========================================================
# 11. SELECCION DE CLASIFICADOS
# =========================================================
def selectQualifiedTeams(groupResults, topPerGroup=2, bestThirdSlots=8):
    qualified = []
    thirdPlacers = []

    for groupName, ranking in groupResults.items():
        qualified.extend(ranking[:topPerGroup])
        if len(ranking) > topPerGroup:
            thirdPlacers.append(ranking[topPerGroup])

    thirdPlacers = sorted(
        thirdPlacers,
        key=lambda x: (x["points"], x["gd"], x["gf"], x["wins"]),
        reverse=True,
    )

    qualified.extend(thirdPlacers[:bestThirdSlots])
    return qualified


# 12. BRACKET DEL RONDA DE 32

In [14]:
# =========================================================
# 12. BRACKET DEL RONDA DE 32
# =========================================================
# Bracket genericamente ordenado por rendimiento.
# Si quieres, esta funcion luego se reemplaza por la tabla oficial exacta
# del fixture de FIFA para el Round of 32.

def buildRoundOf32Pairs(qualifiedTeams):
    ordered = sorted(
        qualifiedTeams,
        key=lambda x: (x["points"], x["gd"], x["gf"], x["wins"], x["group"]),
        reverse=True,
    )

    # Emparejamos mejor vs peor, segundo vs penultimo, etc.
    pairs = []
    n = len(ordered)
    for i in range(n // 2):
        pairs.append((ordered[i]["team"], ordered[n - 1 - i]["team"]))
    return pairs

# 13. SIMULACION DE ELIMINATORIAS

In [15]:
# =========================================================
# 13. SIMULACION DE ELIMINATORIAS
# =========================================================
def simulateKnockoutMatch(homeTeam, awayTeam, teamStates, scaler, poissonHomeModel, poissonAwayModel, drawBoost, rng=None, matchDate=None):
    if rng is None:
        rng = np.random.default_rng()

    match = simulateMatch(
        homeTeam, awayTeam, teamStates, scaler, poissonHomeModel, poissonAwayModel,
        drawBoost=drawBoost, neutral=True, matchDate=matchDate, rng=rng
    )

    if match["homeGoals"] > match["awayGoals"]:
        winner = homeTeam
    elif match["homeGoals"] < match["awayGoals"]:
        winner = awayTeam
    else:
        # Desempate simple por fuerza relativa
        homeStrength = match["lambdaHome"] / max(match["lambdaHome"] + match["lambdaAway"], 1e-9)
        winner = homeTeam if rng.random() < homeStrength else awayTeam

    return {
        "homeTeam": homeTeam,
        "awayTeam": awayTeam,
        "homeGoals": match["homeGoals"],
        "awayGoals": match["awayGoals"],
        "winner": winner,
    }

def simulateKnockoutRound(teamPairs, teamStates, scaler, poissonHomeModel, poissonAwayModel, drawBoost, rng=None, matchDate=None):
    if rng is None:
        rng = np.random.default_rng()

    nextRound = []
    matches = []

    for homeTeam, awayTeam in teamPairs:
        result = simulateKnockoutMatch(
            homeTeam, awayTeam, teamStates, scaler, poissonHomeModel, poissonAwayModel,
            drawBoost=drawBoost, rng=rng, matchDate=matchDate
        )
        nextRound.append(result["winner"])
        matches.append(result)

    return nextRound, matches


# 14. SIMULACION DE UN TORNEO COMPLETO

In [16]:
# =========================================================
# 14. SIMULACION DE UN TORNEO COMPLETO
# =========================================================
def simulateOneTournament(seed=42):
    rng = np.random.default_rng(seed)
    teamStates = deepcopy(currentStates)

    # 1) Play-offs
    uefaWinners = simulateUEFAPlayoffs(
        teamStates, scalerFinal, finalPoissonHome, finalPoissonAway,
        drawBoost=bestConfig["drawBoost"], rng=rng, matchDate=historyDf["date"].max() + pd.Timedelta(days=1)
    )

    interWinners = simulateInterconfedPlayoffs(
        teamStates, scalerFinal, finalPoissonHome, finalPoissonAway,
        drawBoost=bestConfig["drawBoost"], rng=rng, matchDate=historyDf["date"].max() + pd.Timedelta(days=1)
    )

    resolvedGroups = resolveGroupsTemplate(groupsTemplate, uefaWinners, interWinners)

    # 2) Fase de grupos
    groupResults = simulateGroupStage(
        resolvedGroups,
        teamStates,
        scalerFinal,
        finalPoissonHome,
        finalPoissonAway,
        drawBoost=bestConfig["drawBoost"],
        rng=rng,
        matchDate=historyDf["date"].max() + pd.Timedelta(days=1),
    )

    # 3) Clasificados
    qualifiedTeams = selectQualifiedTeams(groupResults, topPerGroup=2, bestThirdSlots=8)

    qualifiedTeamsDf = pd.DataFrame(qualifiedTeams)

    # 4) Ronda de 32
    roundOf32Pairs = buildRoundOf32Pairs(qualifiedTeams)
    round16Teams, _ = simulateKnockoutRound(
        roundOf32Pairs,
        teamStates,
        scalerFinal,
        finalPoissonHome,
        finalPoissonAway,
        drawBoost=bestConfig["drawBoost"],
        rng=rng,
        matchDate=historyDf["date"].max() + pd.Timedelta(days=8),
    )

    # 5) Octavos -> Final
    quarterPairs = buildRoundOf32Pairs([{"team": t, "points": 0, "gd": 0, "gf": 0, "wins": 0, "group": ""} for t in round16Teams])
    quarterTeams, _ = simulateKnockoutRound(
        quarterPairs,
        teamStates,
        scalerFinal,
        finalPoissonHome,
        finalPoissonAway,
        drawBoost=bestConfig["drawBoost"],
        rng=rng,
        matchDate=historyDf["date"].max() + pd.Timedelta(days=15),
    )

    semiPairs = buildRoundOf32Pairs([{"team": t, "points": 0, "gd": 0, "gf": 0, "wins": 0, "group": ""} for t in quarterTeams])
    semiTeams, _ = simulateKnockoutRound(
        semiPairs,
        teamStates,
        scalerFinal,
        finalPoissonHome,
        finalPoissonAway,
        drawBoost=bestConfig["drawBoost"],
        rng=rng,
        matchDate=historyDf["date"].max() + pd.Timedelta(days=22),
    )

    finalPair = [(semiTeams[0], semiTeams[1])] if len(semiTeams) >= 2 else []
    champion = None
    if finalPair:
        finalResult = simulateKnockoutMatch(
            finalPair[0][0],
            finalPair[0][1],
            teamStates,
            scalerFinal,
            finalPoissonHome,
            finalPoissonAway,
            drawBoost=bestConfig["drawBoost"],
            rng=rng,
            matchDate=historyDf["date"].max() + pd.Timedelta(days=29),
        )
        champion = finalResult["winner"]

    return {
        "uefaWinners": uefaWinners,
        "interWinners": interWinners,
        "resolvedGroups": resolvedGroups,
        "groupResults": groupResults,
        "qualifiedTeamsDf": qualifiedTeamsDf,
        "champion": champion,
    }

# 15. MONTE CARLO

In [17]:
# =========================================================
# 15. MONTE CARLO
# =========================================================
def simulateTournamentMonteCarlo(nSimulations=1000, seed=42):
    rng = np.random.default_rng(seed)

    championCounts = defaultdict(int)
    qualifiedCounts = defaultdict(int)

    for sim in range(nSimulations):
        result = simulateOneTournament(seed=int(rng.integers(0, 1_000_000_000)))

        if result["champion"] is not None:
            championCounts[result["champion"]] += 1

        for team in result["qualifiedTeamsDf"]["team"].tolist():
            qualifiedCounts[team] += 1

    summary = pd.DataFrame(
        [
            {
                "team": team,
                "championProb": championCounts[team] / nSimulations,
                "qualifiedProb": qualifiedCounts[team] / nSimulations,
            }
            for team in sorted(set(list(championCounts.keys()) + list(qualifiedCounts.keys())))
        ]
    ).sort_values("championProb", ascending=False).reset_index(drop=True)

    return summary

# 16. EJECUCION

In [20]:
# =========================================================
# 16. EJECUCION
# =========================================================
nSimulations = 2000

simulationSummary = simulateTournamentMonteCarlo(
    nSimulations=nSimulations,
    seed=42,
)

print("Resumen de probabilidades:")
simulationSummary.head(20)

Resumen de probabilidades:


,team,championProb,qualifiedProb
0,Spain,0.1605,0.9845
1,Argentina,0.1535,0.9760
2,Brazil,0.0870,0.9590
3,France,0.0790,0.9100
4,England,0.0565,0.9390
5,Colombia,0.0550,0.9185
6,Portugal,0.0400,0.8665
7,Netherlands,0.0340,0.8565
8,Morocco,0.0315,0.8780
9,Iran,0.0295,0.8885


# 17. EXPORTACION

In [21]:
# =========================================================
# 17. EXPORTACION
# =========================================================
simulationOutputPath = outputDir / "tournament_simulation_summary.csv"
simulationSummary.to_csv(simulationOutputPath, index=False)

print("Resultados guardados en:", simulationOutputPath)

Resultados guardados en: ..\data\output\tournament_simulation_summary.csv
